In [35]:


import sys
import os
sys.path.append(os.path.abspath('..'))

import joblib
import pandas as pd

# Carga el modelo entrenado
model = joblib.load('../models/kmeans_model.pkl')

# Carga los datos de prueba
test_data = pd.read_csv('test_data_temp.csv')


c:\Users\destr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\destr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\destr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator One

In [36]:
# Obtener etiquetas de clúster
clusters = model.predict(test_data)

# Añadir columna con el número de clúster
test_data['cluster'] = clusters


In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import joblib

# --- Cargar modelo con pipeline ---
model = joblib.load('kmeans_model.pkl')  # ya incluye preprocessing + KMeans

# --- Cargar tus datos crudos (mismo formato que entrenaste) ---
test_data = pd.read_csv('test_data.csv')
test_data = test_data.replace(r'^\s*$', np.nan, regex=True)

# --- Aplicar solo el preprocesamiento del pipeline ---
X_processed = model.named_steps['preprocessing'].transform(test_data)

# --- Clustering: predecir clústeres ---
clusters = model.named_steps['clustering'].predict(X_processed)

# --- Reducir dimensiones para visualización ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed)

# --- Graficar los clústeres en 2D ---
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', s=50, alpha=0.7)
plt.title('Visualización de Clústeres (PCA + Pipeline)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.colorbar(scatter, label='Cluster')
plt.grid(True)
plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'kmeans_model.pkl'

In [31]:
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x='cluster', y='C208', data=test_data)  # C208 = Edad
plt.title('Distribución de Edad por Clúster')
plt.grid(True)
plt.show()


ValueError: Could not interpret value `C208` for `y`. An entry with this name does not appear in `data`.

<Figure size 1000x600 with 0 Axes>

In [ ]:
cluster_profile = test_data.groupby('cluster').agg({
    'C208': 'mean',            # Edad promedio
    'INGTRABW': 'median',      # Ingreso mensual
    'C312': lambda x: (x == 3).mean(),  # % informales
    'C333': lambda x: (x == 1).mean(),  # % subempleados
    'C375_1': 'mean'           # % con discapacidad de movimiento
})

cluster_profile.round(3)


TypeError: agg function failed [how->median,dtype->object]

In [40]:
cluster_names = {
    0: "Jóvenes informales",
    1: "Adultos vulnerables",
    2: "Profesionales estables",
    3: "Desempleados crónicos"
}

test_data['perfil'] = test_data['cluster'].map(cluster_names)


In [41]:
# Guardar archivo con clústeres asignados
test_data.to_csv('resultados_clustering_2.csv', index=False)